In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .getOrCreate()


26/04/27 21:01:55 WARN Utils: Your hostname, debian resolves to a loopback address: 127.0.1.1; using 192.168.8.8 instead (on interface wlp2s0)
26/04/27 21:01:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 21:01:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# reading the parquet file
url = '/home/carlos/Dokument/GitHub/de_zoomcamp/w6_batch/code/data/yellow_tripdata_2025-11.parquet'

df = spark.read.parquet(url)

df.show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [ ]:
# Checking current number of partitions
print(df.rdd.getNumPartitions())

8


In [6]:
#repartitioning the dataframe to 4 partitions
df = df.repartition(4)

In [7]:
#saving the dataframe back to repartitioned parquet files
df.write.parquet('/home/carlos/Dokument/GitHub/de_zoomcamp/w6_batch/code/data/yellow/')

In [32]:
# Time-based filtering
# Only trips that started on the 15th of November.
import pyspark.sql.functions as F


In [33]:

trip_count = df.filter(F.col('tpep_pickup_datetime').between('2025-11-15 00:00:00', '2025-11-15 23:59:59')).count()
print(f"Number of trips on November 15th: {trip_count}")

Number of trips on November 15th: 162604


In [34]:
# Trying 
# time_df = df.filter((df.date >= '2023-08-01') & (df.date < '2023-09-01'))

count_trip = df.filter((df.tpep_pickup_datetime >= '2025-11-15 00:00:00') & (df.tpep_pickup_datetime < '2025-11-16 00:00:00')).count()
print(f"Count of trips on November 15th: {count_trip}")

Count of trips on November 15th: 162604


In [40]:
df_time =df.select(
    F.col('tpep_pickup_datetime').alias('pickup_time'), 
    F.col('tpep_dropoff_datetime').alias('dropoff_time'))

In [41]:
df_time.show(5)

+-------------------+-------------------+
|        pickup_time|       dropoff_time|
+-------------------+-------------------+
|2025-11-03 19:01:48|2025-11-03 19:08:48|
|2025-11-10 12:46:18|2025-11-10 13:03:42|
|2025-11-02 14:17:02|2025-11-02 14:25:34|
|2025-11-08 20:05:28|2025-11-08 20:13:12|
|2025-11-02 15:30:08|2025-11-02 15:39:37|
+-------------------+-------------------+
only showing top 5 rows



In [ ]:
df_time = df_time.withColumn(
    'trip_length', 
    F.unix_timestamp('dropoff_time')- F.unix_timestamp('pickup_time')
    )

In [45]:
df_time.show(5)

+-------------------+-------------------+-----------+
|        pickup_time|       dropoff_time|trip_length|
+-------------------+-------------------+-----------+
|2025-11-03 19:01:48|2025-11-03 19:08:48|        420|
|2025-11-10 12:46:18|2025-11-10 13:03:42|       1044|
|2025-11-02 14:17:02|2025-11-02 14:25:34|        512|
|2025-11-08 20:05:28|2025-11-08 20:13:12|        464|
|2025-11-02 15:30:08|2025-11-02 15:39:37|        569|
+-------------------+-------------------+-----------+
only showing top 5 rows



In [60]:
df_time \
    .select(
        'trip_length', 
        F.round(F.col('trip_length')/3600, 2).alias('trip_hours')) \
    .orderBy(F.col('trip_length').desc()) \
    .show(5)

+-----------+----------+
|trip_length|trip_hours|
+-----------+----------+
|     326328|     90.65|
|     277014|     76.95|
|     274370|     76.21|
|     249439|     69.29|
|     241490|     67.08|
+-----------+----------+
only showing top 5 rows



In [83]:
# Joining zones files to the november data
#File already downloaded in the setup step
url = "/home/carlos/Dokument/GitHub/de_zoomcamp/w6_batch/00_setup/taxi_zone_lookup.csv"
zone_df = spark.read.option("header", "true").csv(url)
zone_df.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [ ]:
# name of the LEAST frequent pickup location ID just from the november data
least_freq_pickup_zone = df.groupBy('PULocationID').count().orderBy('count')
least_freq_pickup_zone.show(5, truncate = False)


+------------+-----+
|PULocationID|count|
+------------+-----+
|84          |1    |
|105         |1    |
|5           |1    |
|187         |3    |
|111         |4    |
+------------+-----+
only showing top 5 rows



In [ ]:
# joining with the zone dataframe to get the zone names
df.join(zone_df, df.PULocationID == zone_df.LocationID, 'left') \
.groupBy(['LocationID', 'Zone'])\
.count().orderBy('count').show(5, truncate = False)

+----------+---------------------------------------------+-----+
|LocationID|Zone                                         |count|
+----------+---------------------------------------------+-----+
|105       |Governor's Island/Ellis Island/Liberty Island|1    |
|84        |Eltingville/Annadale/Prince's Bay            |1    |
|5         |Arden Heights                                |1    |
|187       |Port Richmond                                |3    |
|199       |Rikers Island                                |4    |
+----------+---------------------------------------------+-----+
only showing top 5 rows

